# 1. Question Closeness and Decomposition

This notebook does two things:

1. **Question decomposition** — split each market question into a *qualitative semantic core* (the underlying event) and a *quantitative parametric part* (date/time deadlines, numeric thresholds, count conditions). This separation is necessary to distinguish true duplicates from reframings.

2. **Closeness evaluation** — compute pairwise similarity signals and evaluate them against weak labels, including a new reframing-aware taxonomy.

### Closeness taxonomy

| Type | Qualitative similarity | Quantitative part | Example |
|---|---|---|---|
| `EXACT_DUPLICATE` | very high | identical | same question, same deadline |
| `REFRAMING` | very high | different | "end by 7 days" vs "end by 30 days" |
| `RELATED_EVENT` | moderate | varies | same topic, different events |
| `UNRELATED` | low | varies | unrelated questions |

### Why this matters for the benchmark

Side Idea 7 (outdated-prior belief update) requires excluding reframings: we want one canonical question per event family, not multiple parameterized variants of the same underlying question. This notebook provides the tooling to do that split.

## Imports and configuration

In [ ]:
import ast
import itertools
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics.pairwise import cosine_similarity

from polymarket_research.data.canonical import CanonicalDatasetBuilder
from polymarket_research.data.raw import RawExternalCovariates, RawMarketHandle
from polymarket_research.research import QuestionDecoupler
from polymarket_research.utils import setup_root

REPO_ROOT = setup_root()
DATA_SOURCE = 'polymarket'  # or 'kalshi'
ARTEFACT_ROOT = REPO_ROOT / 'frozen_notebooks' / 'running_artefacts' / DATA_SOURCE
CANONICAL_CACHE_DIR = ARTEFACT_ROOT / 'canonical_dataset'

# ── Scope: NO category filter.
# Categories are not used as an input filter here; they are only weak metadata.
MARKET_LIMIT = None
MARKET_ORDER = None  # one of: None, 'latest', 'largest'
MAX_MARKETS = 50000           # cap to keep O(N²) pairwise computation tractable
MIN_PROBABILITY_ROWS = 288  # at least ~1 day of 5-min snapshots
MIN_VOLUME = 1000           # focus on markets with some trading activity
RESAMPLE_FREQ = '1h'
TOP_K = 5

# Closeness taxonomy thresholds
EXACT_DUP_THRESHOLD = 0.95   # qual-core cosine >= this + same quant → EXACT_DUPLICATE
RELATED_THRESHOLD   = 0.40   # qual-core cosine >= this → RELATED_EVENT

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', None)


## Load canonical dataset

We use the canonical resolved-markets layer. If no parquet cache exists, we build from the raw layer.

In [ ]:
if (CANONICAL_CACHE_DIR / 'markets.parquet').exists() and MARKET_LIMIT is None and MARKET_ORDER is None:
    from polymarket_research.data.canonical import CanonicalDataset
    canonical = CanonicalDataset.from_parquet(CANONICAL_CACHE_DIR)
    print('Loaded canonical from parquet cache:', CANONICAL_CACHE_DIR)
else:
    raw_handle = RawMarketHandle(source=DATA_SOURCE)
    raw_bundle = raw_handle.load_bundle(
        include_market_universe=False,
        include_download_manifest=False,
        include_probabilities=True,
        include_raw_trades=False,
        market_limit=MARKET_LIMIT,
        market_order=MARKET_ORDER,
    )
    raw_external = RawExternalCovariates().load()
    canonical = CanonicalDatasetBuilder(
        raw_dataset=raw_bundle,
        raw_external=raw_external,
        resolved_only=True,
    ).build()
    if MARKET_LIMIT is None and MARKET_ORDER is None:
        canonical.save(CANONICAL_CACHE_DIR)
    print('Built canonical from SQLite')

all_markets = canonical.markets.copy()
probabilities = canonical.probabilities.copy()
print(f'Canonical resolved markets: {len(all_markets)}')
print('Data source:', DATA_SOURCE)
print('Market limit:', MARKET_LIMIT)
print('Market order:', MARKET_ORDER)
display(all_markets[['market_id', 'question', 'research_category', 'family_id', 'volume_num']].head(5))


## Filter working scope

In [ ]:
markets = (
    all_markets[
        all_markets['probability_rows'].fillna(0).ge(MIN_PROBABILITY_ROWS)
        & all_markets['volume_num'].fillna(0).ge(MIN_VOLUME)
    ]
    .sort_values('volume_num', ascending=False)
    .head(MAX_MARKETS)
    .reset_index(drop=True)
    .copy()
)

print(f'Working scope: {len(markets)} markets')
print(f'  probability_rows >= {MIN_PROBABILITY_ROWS}: covers {all_markets["probability_rows"].fillna(0).ge(MIN_PROBABILITY_ROWS).sum()} markets in total')
print(f'  volume_num >= {MIN_VOLUME}: covers {all_markets["volume_num"].fillna(0).ge(MIN_VOLUME).sum()} markets in total')
display(markets[["market_id", "question", "volume_num", "probability_rows", "family_id"]].head(10))


---

# Part 1: Question Decomposition

We split each question into:
- **Qualitative core**: the underlying event — who, what, which entity.
- **Quantitative part**: numeric or temporal conditions — deadlines, thresholds, counts.

The decomposition uses regex patterns. We also use structural columns (`event_series_slug`, `group_item_title`, `neg_risk`) as weak validation signals.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Question Decoupling
# ─────────────────────────────────────────────────────────────────

decoupler = QuestionDecoupler()
markets = decoupler.decompose_markets(markets)

display(
    markets[[
        'question',
        'decoupled_event',
        'decoupled_trigger',
        'trigger_candidates',
        'has_trigger',
    ]].head(10)
)


## Validate decomposition with structural columns

We check whether `has_trigger` is consistent with structural markers:
- `neg_risk`: Polymarket's neg-risk mechanism creates parameterized variants.
- `event_series_slug`: markets in the same series often differ only in parameters.
- `group_item_title`: explicitly captures the variant title (e.g. "7 days", "30 days").

In [ ]:
# Retrieve structural columns from raw market_universe if available
struct_cols = ['market_id', 'neg_risk', 'event_series_slug', 'group_item_title']
available_struct_cols = [c for c in struct_cols if c in all_markets.columns]

if len(available_struct_cols) > 1:
    struct = all_markets[available_struct_cols].copy()
    markets = markets.merge(struct, on='market_id', how='left')

    if 'neg_risk' in markets.columns:
        display(Markdown('### has_quant rate by neg_risk flag'))
        display(
            markets.groupby('neg_risk')['has_quant']
            .agg(['mean', 'count'])
            .rename(columns={'mean': 'quant_rate', 'count': 'n'})
        )

    if 'event_series_slug' in markets.columns:
        display(Markdown('### Largest event series'))
        series_sizes = markets.groupby('event_series_slug').size().sort_values(ascending=False).head(10)
        display(series_sizes)

    if 'group_item_title' in markets.columns:
        display(Markdown('### Sample group_item_title (parametric labels)'))
        display(markets[markets['group_item_title'].notna()][['question', 'group_item_title', 'qual_core', 'quant_spans']].head(15))
else:
    print('Structural columns not available in canonical dataset — skipping validation.')

## Decomposition diagnostics

How does the decoupled event differ in length from the original question? How many trigger spans are extracted per market?

In [ ]:
markets['original_len'] = markets['question'].str.len()
markets['qual_len'] = markets['qual_core'].str.len()
markets['quant_char_removed'] = markets['original_len'] - markets['qual_len']
markets['n_quant_spans'] = markets['quant_spans'].apply(len)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.histplot(markets['quant_char_removed'].clip(0, 100), bins=30, color='#4c72b0', ax=axes[0])
axes[0].set_title('Characters removed (quantitative spans)')
axes[0].set_xlabel('Chars removed')

sns.histplot(markets['n_quant_spans'], bins=range(0, 8), discrete=True, color='#dd8452', ax=axes[1])
axes[1].set_title('Number of quantitative spans per question')
axes[1].set_xlabel('Span count')

sns.histplot(markets['qual_len'], bins=30, color='#55a868', ax=axes[2])
axes[2].set_title('Qualitative core length (chars)')
axes[2].set_xlabel('Characters')

plt.tight_layout()
plt.show()

display(Markdown('### Sample: questions with most quantitative content'))
display(
    markets.sort_values('n_quant_spans', ascending=False)[['question', 'qual_core', 'quant_spans']].head(10)
)

---

# Part 2: Pairwise Closeness Signals

We compute:
- **Full-text cosine**: TF-IDF on the original question + description + tags (from existing pipeline).
- **Qual-core cosine**: TF-IDF on the qualitative core only — the key new signal.
- **Tag Jaccard**: overlap of tag sets.
- **Time overlap**: Jaccard of market lifetime windows.
- **Resolution source match**: same resolution URL.
- **Return correlation**: absolute Pearson correlation of hourly probability changes.
- **Combined score**: weighted combination.

In [ ]:
# ─── helpers (reproduced from tickers_closeness for self-containment) ─────────

def parse_listish(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass
    sep = '|' if '|' in text else ','
    return [p.strip() for p in text.split(sep) if p.strip()]


def tag_jaccard(tags_a, tags_b):
    a = set(parse_listish(tags_a))
    b = set(parse_listish(tags_b))
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)


def safe_corr(s_a: pd.Series, s_b: pd.Series, min_points: int = 12) -> float:
    pair = pd.concat([s_a, s_b], axis=1).dropna()
    if len(pair) < min_points:
        return np.nan
    a = pair.iloc[:, 0].to_numpy(float)
    b = pair.iloc[:, 1].to_numpy(float)
    if np.nanstd(a) <= 1e-12 or np.nanstd(b) <= 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def safe_auc(y_true, score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, score)


def safe_ap(y_true, score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, score)


def recall_at_k(pair_df: pd.DataFrame, score_col: str, label_col: str, k: int = 5) -> float:
    recalls = []
    for _, group in pair_df.groupby('market_id_a', sort=False):
        if group[label_col].sum() == 0:
            continue
        top = group.nlargest(k, score_col)
        recalls.append(float(top[label_col].sum() > 0))
    return float(np.mean(recalls)) if recalls else np.nan

In [ ]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
TOKEN_RE = re.compile(r"(?u)\b[\w$%.+-]+\b")

def lemma_tokenizer(text: str) -> list[str]:
    text = str(text or '').lower()
    tokens = TOKEN_RE.findall(text)
    return [lemmatizer.lemmatize(tok) for tok in tokens if len(tok) > 1]

# Start simple: similarity on the raw question text, including quantitative parts.
markets['question_for_similarity'] = markets['question'].fillna('')

vec_question = TfidfVectorizer(
    min_df=2,
    max_features=5000,
    ngram_range=(1, 2),
    tokenizer=lemma_tokenizer,
    preprocessor=None,
    lowercase=False,
    token_pattern=None,
)

tfidf_question = vec_question.fit_transform(markets['question_for_similarity'])
cos_question = cosine_similarity(tfidf_question)   # shape (N, N)

print('TF-IDF matrix built on raw question text')
print(f'  Vocab size: {len(vec_question.vocabulary_)}')
print('  Lemmatization: WordNetLemmatizer')
print('  N-grams: unigrams + bigrams')


In [ ]:
# Visualize text similarity with PCA(2) and cluster with DBSCAN
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Choose which representation to inspect:
# - tfidf_question
# - tfidf_qual
# If you followed the simplified setup, use tfidf_question.
X = tfidf_question

# PCA to 2D for visualization
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X.toarray())

# DBSCAN over the original TF-IDF space.
# For cosine-like behavior, euclidean on l2-normalized TF-IDF is usually acceptable.
dbscan = DBSCAN(
    eps=0.9,
    min_samples=3,
    metric='euclidean',
)
cluster_labels = dbscan.fit_predict(X)

viz = markets[['market_id', 'question']].copy().reset_index(drop=True)
viz['pca_1'] = X_2d[:, 0]
viz['pca_2'] = X_2d[:, 1]
viz['cluster'] = cluster_labels.astype(int)
viz['is_noise'] = viz['cluster'] == -1

print(f'Explained variance by PCA(2): {pca.explained_variance_ratio_.sum():.3f}')
print('Cluster sizes:')
display(viz['cluster'].value_counts().sort_index().rename_axis('cluster').to_frame('n'))

plt.figure(figsize=(14, 10))
sns.scatterplot(
    data=viz,
    x='pca_1',
    y='pca_2',
    hue='cluster',
    style='is_noise',
    palette='tab20',
    alpha=0.8,
    s=70,
)
plt.title('Markets in TF-IDF space: PCA(2) + DBSCAN clusters')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

display(Markdown('### Sample points per cluster'))
for cluster_id in sorted(viz['cluster'].unique()):
    sample = viz[viz['cluster'] == cluster_id][['market_id', 'question']].head(10)
    label = 'noise' if cluster_id == -1 else f'cluster {cluster_id}'
    display(Markdown(f'**{label}**'))
    display(sample.reset_index(drop=True))


In [ ]:
# Build hourly probability returns for return-correlation signal
prob_hourly = (
    probabilities[probabilities['market_id'].isin(markets['market_id'])]
    [['market_id', 'timestamp_utc', 'yes_probability']]
    .dropna()
    .set_index('timestamp_utc')
    .groupby('market_id')['yes_probability']
    .resample(RESAMPLE_FREQ)
    .last()
    .reset_index()
)
wide_prob = prob_hourly.pivot(index='timestamp_utc', columns='market_id', values='yes_probability').sort_index()
wide_ret = wide_prob.diff()
print('Return matrix:', wide_ret.shape)

In [ ]:
# Build pairwise frame
market_ids = markets['market_id'].tolist()
meta = markets.set_index('market_id')

pair_rows = []
for i, j in itertools.combinations(range(len(market_ids)), 2):
    mid_a, mid_b = market_ids[i], market_ids[j]
    row_a, row_b = markets.iloc[i], markets.iloc[j]

    # Temporal overlap
    overlap_start = max(row_a['created_at'], row_b['created_at'])
    overlap_end = min(row_a['end_date'], row_b['end_date'])
    overlap_hours = max(0.0, (overlap_end - overlap_start).total_seconds() / 3600.0)
    union_hours = max(
        (max(row_a['end_date'], row_b['end_date']) - min(row_a['created_at'], row_b['created_at'])).total_seconds() / 3600.0,
        1.0,
    )

    # Return correlation
    corr = np.nan
    if mid_a in wide_ret.columns and mid_b in wide_ret.columns:
        corr = safe_corr(wide_ret[mid_a], wide_ret[mid_b])

    pair_rows.append({
        'market_id_a': mid_a,
        'market_id_b': mid_b,
        'question_a': row_a['question'],
        'question_b': row_b['question'],
        'qual_core_a': row_a['qual_core'],
        'qual_core_b': row_b['qual_core'],
        'quant_spans_a': row_a['quant_spans'],
        'quant_spans_b': row_b['quant_spans'],
        'has_quant_a': row_a['has_quant'],
        'has_quant_b': row_b['has_quant'],
        'research_category_a': row_a['research_category'],
        'research_category_b': row_b['research_category'],
        'family_id_a': row_a['family_id'],
        'family_id_b': row_b['family_id'],
        'text_cosine': float(cos_full[i, j]),
        'qual_cosine': float(cos_qual[i, j]),
        'tag_jaccard': tag_jaccard(row_a.get('tag_labels'), row_b.get('tag_labels')),
        'time_overlap': overlap_hours / union_hours,
        'source_match': float(
            str(row_a.get('resolution_source', '')) == str(row_b.get('resolution_source', ''))
            and pd.notna(row_a.get('resolution_source'))
        ),
        'return_corr': corr,
        # Weak labels
        'same_family': float(row_a['family_id'] == row_b['family_id']),
        'same_research_category': float(row_a['research_category'] == row_b['research_category']),
    })

pairs = pd.DataFrame(pair_rows)
pairs['return_corr_abs'] = pairs['return_corr'].abs()

# Combined score (full text + tags + time + source + return corr)
pairs['combined_score'] = (
    0.40 * pairs['text_cosine'].fillna(0)
    + 0.25 * pairs['qual_cosine'].fillna(0)
    + 0.15 * pairs['tag_jaccard'].fillna(0)
    + 0.10 * pairs['time_overlap'].fillna(0)
    + 0.05 * pairs['source_match'].fillna(0)
    + 0.05 * pairs['return_corr_abs'].fillna(0)
)

print(f'Pairs: {len(pairs):,}')
display(pairs.head(3))


---

# Part 3: Closeness Taxonomy

We assign each pair a type based on qualitative cosine and whether both questions have quantitative spans with different values.

In [ ]:
def _quant_differs(spans_a: list, spans_b: list) -> bool:
    """Return True if both questions have quantitative spans and those spans differ."""
    if not spans_a or not spans_b:
        return False
    return set(spans_a) != set(spans_b)


def classify_pair(row) -> str:
    qc = row['qual_cosine']
    tc = row['text_cosine']
    quant_diff = _quant_differs(row['quant_spans_a'], row['quant_spans_b'])
    if qc >= EXACT_DUP_THRESHOLD:
        if quant_diff:
            return 'REFRAMING'
        else:
            return 'EXACT_DUPLICATE'
    elif qc >= RELATED_THRESHOLD or tc >= RELATED_THRESHOLD:
        return 'RELATED_EVENT'
    else:
        return 'UNRELATED'


pairs['closeness_type'] = pairs.apply(classify_pair, axis=1)

type_counts = pairs['closeness_type'].value_counts().reset_index()
type_counts.columns = ['closeness_type', 'count']
type_counts['share_pct'] = 100.0 * type_counts['count'] / type_counts['count'].sum()
display(type_counts)

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=type_counts, x='closeness_type', y='count', palette='crest', ax=ax)
ax.set_title('Closeness taxonomy: pair counts')
ax.set_xlabel('')
ax.set_ylabel('Pairs')
plt.tight_layout()
plt.show()

## Taxonomy: qualitative examples

Inspect example pairs from each type to verify the taxonomy is sensible.

In [ ]:
for ctype in ['EXACT_DUPLICATE', 'REFRAMING', 'RELATED_EVENT']:
    display(Markdown(f'### {ctype}'))
    subset = pairs[pairs['closeness_type'] == ctype].sort_values('qual_cosine', ascending=False).head(5)
    display(
        subset[['question_a', 'question_b', 'qual_cosine', 'text_cosine', 'quant_spans_a', 'quant_spans_b']]
        .reset_index(drop=True)
    )

## Taxonomy: signal distributions by type

How do the closeness signals distribute across the four types?

In [ ]:
plot_types = ['EXACT_DUPLICATE', 'REFRAMING', 'RELATED_EVENT', 'UNRELATED']
plot_pairs = pairs[pairs['closeness_type'].isin(plot_types)].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title in zip(
    axes,
    ['qual_cosine', 'text_cosine', 'tag_jaccard'],
    ['Qualitative-core cosine', 'Full-text cosine', 'Tag Jaccard'],
):
    sns.boxplot(data=plot_pairs, x='closeness_type', y=col, palette='crest', order=plot_types, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## Key comparison: full-text cosine vs qual-core cosine

The critical question: does the qual-core cosine better separate REFRAMING from EXACT_DUPLICATE pairs, compared to the full-text cosine?

In [ ]:
dup_refr = pairs[pairs['closeness_type'].isin(['EXACT_DUPLICATE', 'REFRAMING'])].copy()
dup_refr['is_reframing'] = (dup_refr['closeness_type'] == 'REFRAMING').astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(
    axes,
    ['text_cosine', 'qual_cosine'],
    ['Full-text cosine (original)', 'Qual-core cosine (new)'],
):
    sns.histplot(
        data=dup_refr, x=col, hue='closeness_type',
        bins=30, stat='density', common_norm=False, ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel(col)

plt.suptitle('Can we distinguish EXACT_DUPLICATE from REFRAMING?', y=1.01)
plt.tight_layout()
plt.show()

if len(dup_refr['is_reframing'].unique()) > 1:
    display(pd.DataFrame([{
        'text_cosine_auroc': safe_auc(dup_refr['is_reframing'], dup_refr['text_cosine'].fillna(0)),
        'qual_cosine_auroc': safe_auc(dup_refr['is_reframing'], dup_refr['qual_cosine'].fillna(0)),
        'n_pairs': len(dup_refr),
    }]))

---

# Part 4: Retrieval Evaluation

We evaluate all signals as retrievers under weak labels (`same_family`, `same_research_category`) and the new `is_reframing` label.


In [ ]:
# Add reframing weak label
pairs['is_reframing'] = (pairs['closeness_type'] == 'REFRAMING').astype(float)
pairs['is_dup_or_refr'] = (pairs['closeness_type'].isin(['EXACT_DUPLICATE', 'REFRAMING'])).astype(float)

score_cols = ['text_cosine', 'qual_cosine', 'tag_jaccard', 'time_overlap', 'source_match', 'return_corr_abs', 'combined_score']
label_cols = ['same_family', 'same_research_category', 'is_reframing', 'is_dup_or_refr']

metric_rows = []
for label_col in label_cols:
    for score_col in score_cols:
        metric_rows.append({
            'label': label_col,
            'score': score_col,
            'roc_auc': safe_auc(pairs[label_col], pairs[score_col].fillna(0)),
            'avg_precision': safe_ap(pairs[label_col], pairs[score_col].fillna(0)),
            f'recall_at_{TOP_K}': recall_at_k(pairs, score_col, label_col, k=TOP_K),
        })

metrics = pd.DataFrame(metric_rows)
display(metrics.sort_values(['label', 'roc_auc'], ascending=[True, False]))


In [ ]:
# Summary plot: ROC-AUC by label and score
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for ax, label_col in zip(axes, label_cols):
    sub = metrics[metrics['label'] == label_col]
    sns.barplot(data=sub, x='roc_auc', y='score', palette='crest', ax=ax)
    ax.set_title(f'ROC-AUC — label: {label_col}')
    ax.set_xlabel('ROC-AUC')
    ax.set_ylabel('')
    ax.set_xlim(0.4, 1.0)

plt.tight_layout()
plt.show()

## Nearest-neighbor inspection

In [ ]:
def nearest_neighbors(pair_df: pd.DataFrame, market_id: str, score_col: str, top_k: int = 5) -> pd.DataFrame:
    left = pair_df.loc[pair_df['market_id_a'] == market_id].copy()
    left = left.rename(columns={'market_id_b': 'neighbor_id', 'question_b': 'neighbor_question', 'closeness_type': 'type'})
    right = pair_df.loc[pair_df['market_id_b'] == market_id].copy()
    right = right.rename(columns={'market_id_a': 'neighbor_id', 'question_a': 'neighbor_question', 'closeness_type': 'type'})
    out = pd.concat([left, right], ignore_index=True)
    cols = ['neighbor_id', 'neighbor_question', 'type', score_col, 'same_family']
    return out[[c for c in cols if c in out.columns]].sort_values(score_col, ascending=False).head(top_k)


# Show neighbors for one example per research category
example_ids = markets.groupby('research_category').head(1)['market_id'].tolist()
for market_id in example_ids:
    question = meta.loc[market_id, 'question']
    print('
' + '=' * 100)
    print('QUERY:', question)
    print('  qual_core:', meta.loc[market_id, 'qual_core'])
    print('  quant_spans:', meta.loc[market_id, 'quant_spans'])
    display(Markdown('**Top-5 by qual_cosine:**'))
    display(nearest_neighbors(pairs, market_id, 'qual_cosine'))
    display(Markdown('**Top-5 by combined_score:**'))
    display(nearest_neighbors(pairs, market_id, 'combined_score'))


## Closeness heatmap

In [ ]:
heatmap_ids = (
    markets.sort_values(['family_id', 'market_id'])
    .head(24)['market_id'].tolist()
)
heat_pairs = pairs[
    pairs['market_id_a'].isin(heatmap_ids) & pairs['market_id_b'].isin(heatmap_ids)
][['market_id_a', 'market_id_b', 'qual_cosine', 'combined_score']].copy()

for col, title in [('qual_cosine', 'Qualitative-core cosine'), ('combined_score', 'Combined score')]:
    heat = pd.DataFrame(np.eye(len(heatmap_ids)), index=heatmap_ids, columns=heatmap_ids)
    for row in heat_pairs.itertuples(index=False):
        heat.loc[row.market_id_a, row.market_id_b] = getattr(row, col)
        heat.loc[row.market_id_b, row.market_id_a] = getattr(row, col)

    plt.figure(figsize=(12, 9))
    sns.heatmap(heat, cmap='mako', square=True, xticklabels=False, yticklabels=False)
    plt.title(f'{title} — sample of {len(heatmap_ids)} markets')
    plt.tight_layout()
    plt.show()


---

# Summary and next steps

### What this notebook produces

1. A `DecomposedQuestion` for every market with `decoupled_event` and `decoupled_trigger` (plus backward-compatible `qual_core` / `quant_spans` aliases).
2. A pairwise closeness frame with signals across text, tags, time, return correlation.
3. A four-way closeness taxonomy: `EXACT_DUPLICATE`, `REFRAMING`, `RELATED_EVENT`, `UNRELATED`.
4. Retrieval metrics under `same_family`, `same_research_category`, and reframing-specific labels.

### Recommended uses downstream

- **Side Idea 7 (outdated-prior benchmark)**: use `closeness_type != 'REFRAMING'` to pick one canonical question per event family.
- **Side Idea 6 (masked market reconstruction)**: use `closeness_type` for group-based train/test splits that respect event family membership.
- **Context retrieval (Idea 3/4)**: replace raw `family_id` with `qual_cosine`-based neighbor lookup; `REFRAMING` pairs provide high-precision training signal for retrieval models.
- **`QuestionDecomposer`**: candidate for promotion to `polymarket_research/utils/text.py` once patterns are validated.
